# 📕 Bài 1 — Hồi quy & cách model học

**Tuần 9**

> Cần chạy trước:
> ```
> pip install -e ".[ml]"
> python curriculum/04-classical-ml/tao_du_lieu.py
> ```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

DATA = Path("../../../data")
if not (DATA / "gia_nha.csv").exists():
    DATA = Path("data")

df = pd.read_csv(DATA / "gia_nha.csv")
print(f"Da nap {len(df)} can nha")
df.head()

## 1. Khảo sát dữ liệu trước (thói quen từ Phase 03)

In [ ]:
print(df.shape)
print()
print(df.isna().sum()[df.isna().sum() > 0])
print()
df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].hist(df["gia"], bins=40, color=MAU_CHINH, edgecolor="white", linewidth=.5)
axes[0].set_title("Phân bố giá (triệu VND)")

axes[1].scatter(df["dien_tich"], df["gia"], alpha=.3, s=14, color=MAU_CHINH)
axes[1].set_xlabel("diện tích (m²)"); axes[1].set_ylabel("giá")
axes[1].set_title("Diện tích vs Giá")

tb = df.groupby("quan")["gia"].mean().sort_values()
axes[2].barh(tb.index, tb.to_numpy(), color=MAU_CHINH, height=.6)
axes[2].set_title("Giá trung bình theo quận")
axes[2].grid(axis="y", visible=False)

plt.tight_layout(); plt.show()

Diện tích rõ ràng liên quan tới giá, nhưng các điểm **loe rộng ra** khi diện tích tăng — dấu hiệu còn biến khác (quận) chi phối. Đó chính là thứ model phải học.

## 2. X và y

In [ ]:
X = df[["dien_tich", "so_phong", "co_thang_may"]]     # tạm bỏ cột có NaN
y = df["gia"]

print("X.shape =", X.shape, "  <- (số mẫu, số đặc trưng)")
print("y.shape =", y.shape)
X.head(3)

## 3. Chia train / test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"train: {len(X_train)}   test: {len(X_test)}")

> `random_state=42` để chia lại vẫn ra y hệt. **Luôn đặt** — nếu không, mỗi lần chạy bạn được một điểm khác và không biết cải tiến có thật hay chỉ là may rủi.

## 4. Ba dòng để train một model

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)          # HỌC
y_pred = model.predict(X_test)       # DỰ ĐOÁN

print("Hệ số học được:")
for ten, he_so in zip(X.columns, model.coef_):
    print(f"  {ten:<16} {he_so:>10.1f}")
print(f"  {'(hệ số chặn)':<16} {model.intercept_:>10.1f}")

**Đọc hệ số:** `dien_tich = 112` nghĩa là *mỗi m² tăng thêm, giá tăng ~112 triệu — khi mọi thứ khác giữ nguyên.*

> ⚠️ **Chỉ so sánh được độ lớn hệ số khi các đặc trưng cùng thang đo.** `dien_tich` (đơn vị m², 25–250) và `co_thang_may` (0/1) không so trực tiếp được.

Ở Phase 02 bạn tự viết gradient descent để tìm những con số này. `.fit()` chỉ là phiên bản tối ưu của chính vòng lặp đó.

## 5. Đánh giá — và BASELINE

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
y_bl = baseline.predict(X_test)

print(f"{'':<12} {'R²':>8} {'MAE':>10} {'RMSE':>10}")
print("-" * 44)
for ten, p in [("Baseline", y_bl), ("Model", y_pred)]:
    print(f"{ten:<12} {r2_score(y_test, p):>8.4f} "
          f"{mean_absolute_error(y_test, p):>10.0f} "
          f"{np.sqrt(mean_squared_error(y_test, p)):>10.0f}")

**Baseline luôn cho R² ≈ 0** — vì nó chính là định nghĩa của R²: so với việc luôn đoán trung bình.

> 🔑 **Luôn có baseline.** Trước khi khoe R² = 0.72, phải trả lời được *"so với cái gì?"*. Rất nhiều dự án ML tiêu hàng trăm giờ để hơn baseline vài phần trăm — và không ai kiểm tra điều đó cho tới khi triển khai xong.

**Luôn báo cáo MAE cùng R².** *"R² = 0.72"* nghe hay nhưng vô hình; *"sai số trung bình 1.6 tỷ trên căn nhà giá trung bình 6 tỷ"* mới cho người nghe hình dung được.

## 6. Nhìn model sai ở đâu

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2)

a1.scatter(y_test, y_pred, alpha=.4, s=18, color=MAU_CHINH)
lims = [y_test.min(), y_test.max()]
a1.plot(lims, lims, color=MAU_NHAN, lw=1.5)
a1.set_xlabel("giá thật"); a1.set_ylabel("giá dự đoán")
a1.set_title("Dự đoán vs Thực tế (đường đỏ = hoàn hảo)")

sai_so = y_test - y_pred
a2.scatter(y_pred, sai_so, alpha=.4, s=18, color=MAU_CHINH)
a2.axhline(0, color=MAU_NHAN, lw=1.5)
a2.set_xlabel("giá dự đoán"); a2.set_ylabel("sai số (thật − dự đoán)")
a2.set_title("Biểu đồ phần dư")

plt.tight_layout(); plt.show()

**Biểu đồ phần dư (residual plot) là công cụ chẩn đoán mạnh nhất của hồi quy.**

| Hình dạng | Nghĩa |
|---|---|
| Rải đều quanh 0, không có hình dạng | ✅ Model ổn |
| Hình **phễu** (loe ra) | Phương sai không đều — thử `log(y)` |
| Hình **cong** | Còn quan hệ phi tuyến chưa nắm được |
| Vài điểm rất xa | Ngoại lai — xem lại chúng là gì |

Ở đây phần dư loe rộng khi giá tăng → model kém chính xác với nhà đắt. Điều đó hợp lý: ta **chưa dùng cột `quan`**, mà quận là yếu tố quyết định giá nhất.

## 7. Thêm biến hạng mục → nhảy vọt

In [ ]:
X2 = pd.get_dummies(df[["dien_tich", "so_phong", "co_thang_may", "quan"]],
                    columns=["quan"], drop_first=True)
print("Sau one-hot:", list(X2.columns))

X2_tr, X2_te, y_tr2, y_te2 = train_test_split(X2, y, test_size=0.2, random_state=42)
m2 = LinearRegression().fit(X2_tr, y_tr2)
p2 = m2.predict(X2_te)

print(f"\nKhông có quận : R² = {r2_score(y_test, y_pred):.4f}   MAE = {mean_absolute_error(y_test, y_pred):,.0f}")
print(f"CÓ quận       : R² = {r2_score(y_te2, p2):.4f}   MAE = {mean_absolute_error(y_te2, p2):,.0f}")

> 💡 **Bài học lớn nhất của classical ML:** thêm một đặc trưng **đúng** cải thiện nhiều hơn hàng giờ tinh chỉnh siêu tham số. Đó là lý do *feature engineering* quan trọng hơn *model selection*.

## 8. ⭐ Overfitting — nhìn tận mắt

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
x_nho = np.sort(rng.uniform(-3, 3, 25))
y_nho = 0.5 * x_nho**2 + x_nho + rng.normal(0, 1.2, 25)

x_muot = np.linspace(-3.2, 3.2, 300)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

for ax, bac in zip(axes, [1, 2, 15]):
    mo = make_pipeline(PolynomialFeatures(bac), LinearRegression())
    mo.fit(x_nho.reshape(-1, 1), y_nho)
    ax.scatter(x_nho, y_nho, s=40, color=MAU_CHINH, zorder=3)
    ax.plot(x_muot, mo.predict(x_muot.reshape(-1, 1)), color=MAU_NHAN, lw=2)
    ax.set_ylim(-4, 10)
    r2 = r2_score(y_nho, mo.predict(x_nho.reshape(-1, 1)))
    nhan = {1: "UNDERFIT", 2: "VỪA ĐỦ", 15: "OVERFIT"}[bac]
    ax.set_title(f"Bậc {bac} — {nhan}\nR² trên train = {r2:.3f}", fontsize=10)

plt.tight_layout(); plt.show()

Bậc 15 có **R² trên train cao nhất** — nó đi qua gần như mọi điểm. Nhưng nhìn đường cong: nó uốn lượn điên rồ ở giữa các điểm. Đó là học thuộc **nhiễu**, không phải học **quy luật**.

Giờ đo trên dữ liệu **model chưa từng thấy**:

In [ ]:
x_tr, x_te, yt, ye = train_test_split(
    x_nho.reshape(-1, 1), y_nho, test_size=0.3, random_state=1
)

print(f"{'bậc':>4} {'R² train':>10} {'R² test':>10}   chẩn đoán")
print("-" * 48)
for bac in [1, 2, 3, 5, 9, 15]:
    mo = make_pipeline(PolynomialFeatures(bac), LinearRegression()).fit(x_tr, yt)
    rtr = r2_score(yt, mo.predict(x_tr))
    rte = r2_score(ye, mo.predict(x_te))
    cd = "Overfit" if rtr - rte > 0.1 else ("Underfit" if rtr < 0.5 else "Tốt")
    print(f"{bac:>4} {rtr:>10.4f} {rte:>10.4f}   {cd}")

```
      Điểm số
        │        ╭─────────── train  (luôn tăng khi model phức tạp hơn)
        │      ╭─╯
        │    ╭─╯    ╭──╮
        │  ╭─╯   ╭──╯   ╰──╮
        │╭─╯  ╭──╯          ╰────── test  (tăng rồi GIẢM)
        └──────┬──────────────────▶ độ phức tạp
            điểm ngọt
```

> 🔑 **Đây là biểu đồ quan trọng nhất của machine learning.** Điểm train luôn tăng khi model phức tạp hơn; điểm test tăng rồi **giảm**. Việc của bạn là tìm đỉnh của đường test.

**Bảng chẩn đoán:**

| Train | Test | Kết luận | Chữa |
|---|---|---|---|
| Dở | Dở | Underfit | Model phức tạp hơn, thêm đặc trưng |
| Tốt | Tốt | ✅ Vừa | — |
| Rất tốt | Dở | Overfit | Nhiều dữ liệu hơn, model đơn giản hơn, regularization |
| Dở | Tốt | 🚨 Có bug | Kiểm tra lại cách chia dữ liệu |

## 9. Regularization — chữa overfitting

In [ ]:
from sklearn.linear_model import Ridge

print(f"{'alpha':>8} {'R² train':>10} {'R² test':>10}")
print("-" * 30)
for alpha in [0, 0.01, 1, 100, 10_000]:
    mo = make_pipeline(PolynomialFeatures(15), Ridge(alpha=max(alpha, 1e-10)))
    mo.fit(x_tr, yt)
    print(f"{alpha:>8} {r2_score(yt, mo.predict(x_tr)):>10.4f} "
          f"{r2_score(ye, mo.predict(x_te)):>10.4f}")

**Ridge** thêm một khoản phạt vào loss: `loss + alpha × Σw²`. Model bị buộc phải giữ hệ số nhỏ, nên không thể uốn lượn điên cuồng nữa.

| `alpha` | Tác dụng |
|---|---|
| 0 | Không phạt → hồi quy tuyến tính thường → vẫn overfit |
| Vừa | Điểm test tốt nhất ✅ |
| Rất lớn | Ép hết hệ số về ~0 → underfit |

| Loại | Phạt | Đặc điểm |
|---|---|---|
| **Ridge (L2)** | `Σw²` | Thu nhỏ hệ số, giữ tất cả |
| **Lasso (L1)** | `Σ\|w\|` | Đưa hệ số về đúng 0 → **tự chọn đặc trưng** |
| **ElasticNet** | cả hai | Kết hợp |

> 🔗 Ý tưởng "phạt model phức tạp" quay lại ở Phase 05 dưới dạng **dropout** và **weight decay**.

## 10. Tổng kết

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)   # ① LUÔN có
model = LinearRegression().fit(X_train, y_train)                    # ② train

r2_score(y_test, model.predict(X_test))                             # ③ đo
mean_absolute_error(y_test, model.predict(X_test))                  # ④ và đo bằng đơn vị thật
```

**Bốn thói quen mang theo cả sự nghiệp:**
1. Luôn có **baseline**
2. Luôn so **train vs test** để phát hiện overfit
3. Luôn báo cáo metric **bằng đơn vị gốc** (MAE), không chỉ R²
4. Luôn **vẽ phần dư** — nó lộ ra thứ mà con số giấu đi

## ✍️ Bài tập

```bash
pytest tests/phase04/test_ex01.py -v
```